# docTR Setup & Testing

In [2]:
%pip install python-doctr

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install "python-doctr[viz,html,contrib]"

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -e doctr/.

Obtaining file:///Users/typunleu/Documents/ACADEMIC/CADT/CADT%20Y3/Y3%20Course%20Materials/Y3T2_Course_Material%20%28Active%29/Capstone%20Project%20/Capstone%20Materials/docTR_testing/doctr
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for python-doctr (pyproject.toml) ... done
  Created wheel for python-doctr: filename=python_doctr-1.0.2a0-0.editable-py3-none-any.whl size=17979 sha256=54ec9f7aeab096e217b238185627e19704a41e4c67c22fba01fa4478aab0c12a
  Stored in directory: /private/var/folders/_y/gjt901gx33b3cdzq9gyhjxnr0000gn/T/pip-ephem-wheel-cache-uvxvpgg5/wheels/9f/1a/52/d42cb0d7a1b863b855c00a1c3d600ee3edd5807690e16cb1de
Successfully built python-doctr
  Attempting uninstall: python-doctr
    Found existing installation: python-doctr 1.0.2a0
    Uninstalling python-doctr-1.0.2a0:
      Successfully unins

In [ ]:
from doctr.models import ocr_predictor, crnn_vgg16_bn
from doctr.datasets import VOCABS
from doctr.io import DocumentFile
from tqdm import tqdm
import time
import json

# 1) Build Khmer-aware OCR model
print("Loading OCR model...")
reco_model = crnn_vgg16_bn(pretrained=True, vocab=VOCABS["khmer"])
model = ocr_predictor(
    det_arch="fast_base",
    reco_arch=reco_model,   # use reco_arch, not reco_model
    pretrained=True,
    detect_language=True
)
print("Model loaded successfully.")

# 2) Read image
print("Reading image...")
doc = DocumentFile.from_images("demo_hand_written.jpg")

# 3) Progress bar (visual only)
print("Running OCR...")
for _ in tqdm(range(5), desc="Processing", ncols=100):
    time.sleep(0.3)

# 4) Run OCR
result = model(doc)

# 5) Show detected boxes + text overlay
result.show()

# 6) Export OCR structure
ocr_data = result.export()

# Save structured result
with open("ocr_result.json", "w", encoding="utf-8") as f:
    json.dump(ocr_data, f, indent=2, ensure_ascii=False)

# 7) Extract plain text
all_pages_text = []
for page in ocr_data["pages"]:
    page_lines = []
    for block in page["blocks"]:
        for line in block["lines"]:
            line_text = " ".join(word["value"] for word in line["words"])
            page_lines.append(line_text)
    all_pages_text.append("\n".join(page_lines))

full_text = "\n\n--- PAGE BREAK ---\n\n".join(all_pages_text)

# Save plain text
with open("ocr_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

# 8) Output everything at once
print("\n========== OCR EXTRACTED TEXT ==========\n")
print(full_text if full_text.strip() else "[No text recognized]")

# Optional language output (if available)
print("\n========== PAGE LANGUAGE ==========")
for i, page in enumerate(ocr_data["pages"], start=1):
    lang = page.get("language")
    if lang:
        print(f"Page {i}: {lang.get('value')} (confidence={lang.get('confidence'):.3f})")
    else:
        print(f"Page {i}: [No language prediction]")

print("\nSaved:")
print("- ocr_result.json")
print("- ocr_text.txt")
